## Importing libraries and datasets

In [1]:
from colorsys import yiq_to_rgb
from xml.etree.ElementInclude import include

import pandas as pd
import numpy as np
from flask.app import T_template_test
from tensorflow.python.tpu.ops.gen_xla_ops import \
    xla_sparse_dense_matmul_grad_with_ftrl_and_csr_input
from tensorflow.python.util.nest_util import yield_value


In [2]:
df = pd.read_csv("/Users/lata.bharati/Desktop/Projects/customer-churn-prediction/data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

## Preparing target and features

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [4]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [5]:
df["Churn"].value_counts()


Churn
0    5174
1    1869
Name: count, dtype: int64

Remove Customer ID

Because it has no meaningful predictive information

In [8]:
X = df.drop(columns= ["customerID", "Churn"])
y = df["Churn"]

Treating Senior Citizen as categorical

In [9]:
X["SeniorCitizen"] = X["SeniorCitizen"].astype(str)

In [14]:
categorical_features = X.select_dtypes(include = ["object", "string"]).columns.tolist()
numerical_features = X.select_dtypes(include = ["int64", "float64"]).columns.tolist()

print("Numerical features: ", numerical_features)
print("Categorical features: ", categorical_features)

Numerical features:  ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features:  ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## Splitting in training-validation-test sets

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size = 0.3, stratify=y, random_state = 42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size = 0.5, stratify=y_temp, random_state = 42)

In [16]:
print("Training set size: ", X_train.shape)
print("Validation set size: ", X_val.shape)
print("Test set size: ", X_test.shape)

Training set size:  (4930, 19)
Validation set size:  (1056, 19)
Test set size:  (1057, 19)


Checking churn proportion in various sets

In [17]:
for name, target in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(name, round(target.mean() *100, 2),"%")

Train 26.53 %
Validation 26.52 %
Test 26.58 %


## Preprocessing pipelines